In [ ]:
import lime
import lime.lime_tabular
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from imblearn.combine import SMOTETomek
from imblearn.over_sampling import ADASYN
import warnings

warnings.filterwarnings('ignore')

# Load Data
df = pd.read_csv('heart.csv')
le = LabelEncoder()
for col in ['Sex', 'ChestPainType', 'RestingECG', 'ExerciseAngina', 'ST_Slope']:
    df[col] = le.fit_transform(df[col])

X = df.drop('HeartDisease', axis=1)
y = df['HeartDisease']
feature_names = X.columns.tolist()

# Split (Seed 369)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=369)

# Scale
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Balance Data
smt = SMOTETomek(random_state=369)
X_res_lgbm, y_res_lgbm = smt.fit_resample(X_train_scaled, y_train)

ada = ADASYN(random_state=369)
X_res_xgb, y_res_xgb = ada.fit_resample(X_train_scaled, y_train)

# Train SOTA Models (Locked Parameters)
model_lgbm = LGBMClassifier(learning_rate=0.2094, num_leaves=29, random_state=369, verbose=-1)
model_lgbm.fit(X_res_lgbm, y_res_lgbm)

model_xgb = XGBClassifier(learning_rate=0.1562, max_depth=3, use_label_encoder=False, eval_metric='logloss', random_state=369)
model_xgb.fit(X_res_xgb, y_res_xgb)

# DEFINE ENSEMBLE PREDICTION WRAPPER
def ensemble_predict_proba(data_array):
    # LIME passes a numpy array. We predict with both models and weight them.
    prob_lgbm = model_lgbm.predict_proba(data_array)
    prob_xgb = model_xgb.predict_proba(data_array)

    # Apply AIW-GTO Weights: 0.22 LGBM + 0.78 XGB
    weighted_probs = (prob_lgbm * 0.22) + (prob_xgb * 0.78)
    return weighted_probs

# SETUP LIME EXPLAINER

print(" Initializing LIME Explainer...")

explainer = lime.lime_tabular.LimeTabularExplainer(
    training_data=np.array(X_train_scaled),
    feature_names=feature_names,
    class_names=['Normal', 'Heart Disease'],
    mode='classification',
    random_state=369
)

# GENERATE EXPLANATION FOR A SPECIFIC PATIENT
# Picking patient #15
patient_idx = 15
patient_data_scaled = X_test_scaled[patient_idx]
true_label = y_test.iloc[patient_idx]
true_status = "Heart Disease" if true_label == 1 else "Normal"

print(f"--- Explaining Patient #{patient_idx} ---")
print(f"Actual Truth: {true_status}")

# Generate Explanation using the ensemble wrapper
exp = explainer.explain_instance(
    data_row=patient_data_scaled,
    predict_fn=ensemble_predict_proba,
    num_features=10  # Show top 10 features
)

exp.save_to_file(f'LIME_Explanation_Patient_{patient_idx}.html')

#  GENERATE LIME PLOT
print("Generating LIME Bar Plot...")

# Extract data for plotting
explanation_data = exp.as_list()
feat_names = [x[0] for x in explanation_data]
feat_vals = [x[1] for x in explanation_data]

colors = ['#FF6F61' if x > 0 else '#48C9B0' for x in feat_vals]

# Create Plot
plt.figure(figsize=(10, 6), dpi=300)
plt.barh(feat_names, feat_vals, color=colors, edgecolor='white', height=0.7)

# Styling
plt.title(f"LIME Analysis: Patient #{patient_idx} (True: {true_status})", fontsize=14, fontweight='bold', pad=20)
plt.xlabel("Contribution to Prediction Probability", fontsize=12)
plt.axvline(0, color='grey', linewidth=0.8, linestyle='--')
plt.grid(axis='x', linestyle=':', alpha=0.6)

# Custom Legend
legend_elements = [
    Patch(facecolor='#FF6F61', label='Increases Risk (Heart Disease)'),
    Patch(facecolor='#48C9B0', label='Decreases Risk (Normal)')
]
plt.legend(handles=legend_elements, loc='lower right', fontsize=10)

plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig(f'LIME_Patient_{patient_idx}_Analysis.png', dpi=300)
plt.show()

print(f"\nProcess Complete.")